# End-to-End Pipeline Test

Exercises every service built so far in a single linear pipeline:

1. **Arxiv client** → fetch paper metadata
2. **PDF parser (Docling)** → download + parse PDF
3. **Repository (Postgres)** → persist metadata
4. **TextChunker** → split parsed text into chunks
5. **OpenAI embeddings** → vectorize each chunk
6. **OpenSearch client** → create index, bulk-write chunks
7. **Search** → BM25, vector, and hybrid queries

Run cells top to bottom. Each cell prints what it produced so you can verify before continuing.

**Prerequisites:**
- Postgres + Airflow containers running (`docker compose up -d postgres airflow`).
- OpenSearch container running (`docker compose up -d opensearch`).
- `OPENAI_API_KEY` in `.env`.

## 0. Setup

Locate the project root, add it to `sys.path`, configure logging so we can see what services are doing.

In [1]:
import logging
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    force=True,
)

Project root: /Users/kumarrohit/Arxiv_Paper_Curator


## 1. Build all services

Every service is constructed via its factory. Each factory is `@lru_cache`d, so calling these is idempotent and cheap.

Important: import `Paper` **before** `make_database()` so the ORM table registers against `Base` before `create_all` runs.

In [2]:
from src.models.paper import Paper  # MUST be imported before make_database()
from src.db.factory import make_database
from src.services.arxiv.factory import make_arxiv_client
from src.services.pdf_parser.factory import make_pdf_parser_service
from src.services.opensearch.factory import make_opensearch_client
from src.services.embeddings.factory import make_openai_embeddings_client
from src.services.indexing.text_chunker import TextChunker
from src.repositories.paper import PaperRepository

db        = make_database()
arxiv     = make_arxiv_client()
parser    = make_pdf_parser_service()       # first call downloads Docling models (~1GB, slow!)
opensearch = make_opensearch_client()
embedder  = make_openai_embeddings_client()
chunker   = TextChunker()                    # no factory yet — built directly

print("All services built.")

2026-06-07 16:10:46,694 | INFO    | src.db.interfaces.postgresql | Attempting to connect to PostgreSQL at: localhost:5432/rag_db
2026-06-07 16:10:46,942 | INFO    | src.db.interfaces.postgresql | Database connection test successful
2026-06-07 16:10:46,958 | INFO    | src.db.interfaces.postgresql | All tables already exist - no new tables created
2026-06-07 16:10:46,959 | INFO    | src.db.interfaces.postgresql | PostgreSQL database initialized successfully
2026-06-07 16:10:46,959 | INFO    | src.db.interfaces.postgresql | Database: rag_db
2026-06-07 16:10:46,959 | INFO    | src.db.interfaces.postgresql | Total tables: papers
2026-06-07 16:10:46,959 | INFO    | src.db.interfaces.postgresql | Database connection established
2026-06-07 16:10:47,001 | INFO    | src.services.opensearch.client | OpenSearch client initialized with host: http://localhost:9200
2026-06-07 16:10:47,027 | INFO    | src.services.embeddings.openai_client | OpenAI embeddings client initialized: model=text-embedding-3-

All services built.


## 2. Health checks

Confirm each external system is reachable before doing real work. If any of these fail, fix that before continuing.

In [4]:
from sqlalchemy import text

# Postgres
with db.get_session() as session:
    pg_ok = session.execute(text("SELECT 1")).scalar() == 1
print(f"Postgres reachable: {pg_ok}")

# OpenSearch
print(f"OpenSearch healthy: {opensearch.health_check()}")

# OpenAI
vec = await embedder.embed_text("healthcheck")
print(f"OpenAI reachable: {len(vec) == embedder.settings.dimensions} ({len(vec)} dims)")

Postgres reachable: True


2026-06-07 16:10:47,333 | INFO    | opensearch | GET http://localhost:9200/_cluster/health [status:200 request:0.262s]


OpenSearch healthy: True


2026-06-07 16:10:49,377 | INFO    | httpx | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-07 16:10:49,388 | INFO    | src.services.embeddings.openai_client | Embedded 1 texts in 1 batch(es)


OpenAI reachable: True (1024 dims)


## 3. Setup OpenSearch index + RRF pipeline

Creates the `arxiv-papers-chunks` index (with the hybrid mapping) and registers the RRF search pipeline. Idempotent — safe to re-run.

In [5]:
setup_results = opensearch.setup_indices(force=False)
print("Setup results:", setup_results)
print("Index stats:  ", opensearch.get_index_stats())

2026-06-07 16:10:52,021 | INFO    | opensearch | PUT http://localhost:9200/arxiv-papers-chunks [status:200 request:2.087s]
2026-06-07 16:10:52,030 | INFO    | src.services.opensearch.client | Created hybrid index: arxiv-papers-chunks
2026-06-07 16:10:52,044 | WARNING | opensearch | GET http://localhost:9200/_ingest/pipeline/hybrid-rrf-pipeline [status:404 request:0.012s]
2026-06-07 16:10:52,140 | INFO    | opensearch | PUT http://localhost:9200/_search/pipeline/hybrid-rrf-pipeline [status:200 request:0.095s]
2026-06-07 16:10:52,140 | INFO    | src.services.opensearch.client | Created RRF search pipeline: hybrid-rrf-pipeline
2026-06-07 16:10:52,202 | INFO    | opensearch | HEAD http://localhost:9200/arxiv-papers-chunks [status:200 request:0.062s]
2026-06-07 16:10:52,257 | INFO    | opensearch | GET http://localhost:9200/arxiv-papers-chunks/_stats [status:200 request:0.054s]


Setup results: {'hybrid_index': True, 'rrf_pipeline': True}
Index stats:   {'index_name': 'arxiv-papers-chunks', 'exists': True, 'document_count': 0, 'deleted_count': 0, 'size_in_bytes': 208}


## 4. Fetch one paper from arxiv

Calls the arxiv API and gets recent cs.AI papers. Limited to 1 so we don't burn through rate limits.

In [11]:
papers = await arxiv.fetch_papers(max_results=10)
if not papers:
    raise RuntimeError("No papers returned — arxiv may be rate-limiting. Wait a few minutes.")

print(f"Got {len(papers)} papers from arxiv:\n")
for i, p in enumerate(papers):
    print(f"  [{i}] {p.arxiv_id}  ({len(p.authors)} authors)  {p.title[:70]}")


2026-06-07 16:14:34,207 | INFO    | src.services.arxiv.client | Fetching 1 cs.AI papers from arxiv
2026-06-07 16:14:34,765 | INFO    | httpx | HTTP Request: GET https://export.arxiv.org/api/query?search_query=cat:cs.AI&start=0&max_results=1&sortBy=submittedDate&sortOrder=descending "HTTP/1.1 200 OK"
2026-06-07 16:14:34,782 | INFO    | src.services.arxiv.client | Fetched 1 papers.


arxiv_id : 2606.06493v1
title    : HANDOFF: Humanoid Agentic Task-Space Whole-Body Control via Distilled Complementary Teachers
authors  : ['Lizhi Yang', 'Junheng Li', 'Nehar Poddar'] ...
published: 2026-06-04T17:59:50Z
pdf_url  : https://arxiv.org/pdf/2606.06493v1


/var/folders/bn/q6q84kl961g01gkjcnxj1jk80000gn/T/ipykernel_61586/4161460935.py:1: RuntimeWarning: coroutine 'ArxivClient.fetch_papers' was never awaited
  papers = await arxiv.fetch_papers(max_results=1)


## 5. Download + parse the PDF

Downloads the PDF (cached locally on second run), then Docling extracts structured content. **First run is slow** — Docling downloads model weights.

In [ ]:
from src.exceptions import PDFParsingException, PDFValidationError

paper = None
pdf_path = None
parsed = None

for i, candidate in enumerate(papers):
    print(f"\n[{i}] Trying {candidate.arxiv_id} — {candidate.title[:60]}...")
    try:
        pdf_path = await arxiv.download_pdf(candidate)
        parsed = await parser.parse_pdf(pdf_path)
        paper = candidate
        print(f"  ✓ Parsed: {len(parsed.raw_text)} chars, {len(parsed.sections)} sections")
        break
    except (PDFParsingException, PDFValidationError, Exception) as e:
        print(f"  ✗ Failed: {type(e).__name__}: {e}")
        continue

if paper is None:
    raise RuntimeError("None of the candidate papers could be parsed. Try again later or bump PDFParserSettings limits.")

print(f"\nUsing paper: {paper.arxiv_id} — {paper.title}")
for s in parsed.sections[:5]:
    print(f"  - {s.title!r:50}  ({len(s.content.split())} words)")


PDF saved to: <coroutine object ArxivClient.download_pdf at 0x136335700>


/Users/kumarrohit/Arxiv_Paper_Curator/.venv/lib/python3.12/site-packages/pygments/style.py:66: RuntimeWarning: coroutine 'ArxivClient.fetch_papers' was never awaited
  def colorformat(text):


AttributeError: 'coroutine' object has no attribute 'raw_text'

## 6. Store in Postgres

Use the repository's `upsert` — idempotent, so re-running this cell with the same paper updates instead of duplicating.

In [ ]:
from datetime import datetime
from src.schemas.arxiv.paper import PaperCreate

paper_create = PaperCreate(
    arxiv_id=paper.arxiv_id,
    title=paper.title,
    authors=paper.authors,
    abstract=paper.abstract,
    categories=paper.categories,
    published_date=datetime.fromisoformat(paper.published_date.replace("Z", "+00:00")),
    pdf_url=paper.pdf_url,
    raw_text=parsed.raw_text,
    sections=[s.model_dump() for s in parsed.sections],
    parser_used="docling",
    pdf_processed=True,
    pdf_processing_date=datetime.utcnow(),
)

with db.get_session() as session:
    repo = PaperRepository(session)
    stored = repo.upsert(paper_create)
    stored_id = stored.id  # capture before session closes
    total = repo.get_count()

print(f"Stored paper id: {stored_id}")
print(f"Total papers in DB: {total}")

## 7. Chunk the paper

Use the section-aware chunker. With sections present it uses the hybrid strategy; without sections it falls back to word-based chunking.

In [ ]:
sections_dict = {s.title: s.content for s in parsed.sections}

chunks = chunker.chunk_paper(
    title=paper.title,
    abstract=paper.abstract,
    full_text=parsed.raw_text,
    arxiv_id=paper.arxiv_id,
    paper_id=str(stored_id),
    sections=sections_dict,
)

print(f"Produced {len(chunks)} chunks")
for i, c in enumerate(chunks[:5]):
    print(f"  chunk {i}: {c.metadata.word_count:>4} words  |  section: {c.metadata.section_title!r}")
print(f"  ... and {max(0, len(chunks) - 5)} more")

## 8. Embed the chunks

Single `embed_batch` call — auto-batches into groups of `settings.batch_size` (default 100). For ~20 chunks that's 1 API call.

Costs roughly **$0.0001** for a typical paper.

In [ ]:
chunk_texts = [c.text for c in chunks]
vectors = await embedder.embed_batch(chunk_texts))

print(f"Embedded {len(vectors)} chunks")
print(f"Each vector has {len(vectors[0])} dimensions")
print(f"First vector starts: {vectors[0][:4]}")
assert len(vectors) == len(chunks, "Embed count must match chunk count!"

## 9. Index the chunks in OpenSearch

Convert each (`TextChunk`, `embedding`) into the dict shape `bulk_index_chunks` expects. We fill in the metadata fields the index mapping requires.

In [ ]:
from datetime import datetime

chunks_for_indexing = []
now_iso = datetime.utcnow().isoformat()

for chunk, vector in zip(chunks, vectors):
    chunk_data = {
        "chunk_id":         f"{paper.arxiv_id}_{chunk.metadata.chunk_index}",
        "arxiv_id":         paper.arxiv_id,
        "paper_id":         str(stored_id),
        "chunk_index":      chunk.metadata.chunk_index,
        "chunk_text":       chunk.text,
        "chunk_word_count": chunk.metadata.word_count,
        "start_char":       chunk.metadata.start_char,
        "end_char":         chunk.metadata.end_char,
        "title":            paper.title,
        "authors":          ", ".join(paper.authors),
        "abstract":         paper.abstract,
        "categories":       paper.categories,
        "published_date":   paper.published_date,
        "section_title":    chunk.metadata.section_title or "unknown",
        "embedding_model":  embedder.settings.model,
        "created_at":       now_iso,
        "updated_at":       now_iso,
    }
    chunks_for_indexing.append({"chunk_data": chunk_data, "embedding": vector})

result = opensearch.bulk_index_chunks(chunks_for_indexing)
print("Bulk index result:", result)
print("Index stats now:  ", opensearch.get_index_stats())

## 10. Search the index — BM25 (keyword)

Pure keyword search. Pick a word that should appear in the paper's content.

In [ ]:
# Use a word from the paper's title so we know it should match.
keyword = paper.title.split()[0]
print(f"Searching for keyword: {keyword!r}\n")

results = opensearch.search_papers(query=keyword, size=3)
print(f"Total hits: {results['total']}")
for h in results["hits"]:
    print(f"  score={h['score']:.3f}  section={h.get('section_title')}")
    print(f"  text: {h['chunk_text'][:120]}...")
    print()

## 11. Search the index — pure vector (semantic)

Embed a natural-language query, then ask OpenSearch for nearest-neighbor chunks by vector similarity.

In [ ]:
query = "main contribution of this paper"
query_vec = await embedder.embed_text(query))

results = opensearch.search_chunks_vector(query_embedding=query_vec, size=3)
print(f"Query: {query!r}")
print(f"Total hits: {results['total']}\n")
for h in results["hits"]:
    print(f"  score={h['score']:.3f}  section={h.get('section_title')}")
    print(f"  text: {h['chunk_text'][:120]}...")
    print(

## 12. Search the index — hybrid (BM25 + vector + RRF)

The headline feature. Same query, but OpenSearch runs BOTH retrievers and fuses the rankings via the RRF pipeline.

In [ ]:
query = "main contribution of this paper"
query_vec = await embedder.embed_text(query))

results = opensearch.search_chunks_hybrid(
    query=query,
    query_embedding=query_vec,
    size=3,
)
print(f"Query: {query!r}")
print(f"Total hits: {results['total']}\n")
for h in results["hits"]:
    print(f"  score={h['score']:.4f}  section={h.get('section_title')}")
    print(f"  text: {h['chunk_text'][:120]}...")
    print(

## 13. (Optional) Inspect everything you wrote

Lists Postgres rows and OpenSearch chunks for the paper you just processed. Useful for sanity-checking that all stages stored what they should have.

In [ ]:
print("=== Postgres ===")
with db.get_session() as session:
    repo = PaperRepository(session)
    row = repo.get_by_arxiv_id(paper.arxiv_id)
    print(f"  arxiv_id={row.arxiv_id}  pdf_processed={row.pdf_processed}  sections={len(row.sections or [])}")

print("\n=== OpenSearch ===")
os_chunks = opensearch.get_chunks_by_paper(paper.arxiv_id)
print(f"  Found {len(os_chunks)} chunks for {paper.arxiv_id}")
for c in os_chunks[:3]:
    print(f"    chunk_index={c['chunk_index']}  section={c.get('section_title')}  words={c.get('chunk_word_count')}")

## 14. (Optional) Cleanup

Removes the chunks you just indexed from OpenSearch (Postgres row is left alone — you can re-upsert anytime). Useful when you're iterating and don't want stale data.

In [ ]:
deleted = opensearch.delete_paper_chunks(paper.arxiv_id)
print(f"Deleted chunks for {paper.arxiv_id}: {deleted}")
print("Stats now:", opensearch.get_index_stats())

## What this notebook proved

If every cell ran without errors and the search cells returned the paper's chunks, then the full pipeline works:

```
arxiv API  →  ArxivClient.fetch_papers()        ✓
           →  ArxivClient.download_pdf()         ✓
           →  PDFParserService.parse_pdf()       ✓  (Docling)
           →  PaperRepository.upsert()           ✓  (Postgres)
           →  TextChunker.chunk_paper()          ✓
           →  OpenAIEmbeddingsClient.embed_batch() ✓
           →  OpenSearchClient.bulk_index_chunks() ✓
           →  OpenSearchClient.search_*()        ✓  (BM25, vector, hybrid)
```

**What's NOT in this notebook yet:**
- The Airflow DAG task that wraps all this (the empty `indexing.py`).
- A `HybridIndexingService` that composes the chunker + embedder + opensearch into one `index_paper(paper)` method.

Both are mechanical wrap-ups of what this notebook already does — no new concepts. Once they're built, the DAG can run this whole flow per paper automatically.